# Как Natasha выделяет сущности

На двух фразах из судебного акта `test_002184` посмотрим, как Natasha
разбивает текст на токены и как модель Slovnet возвращает найденные
сущности. Затем посмотрим один объект из сохранённого ответа YandexGPT
на полный акт.

Для повторного запуска нужен корпус, загруженный по инструкции в README.
Установите Natasha в окружение текущего ядра, если её там ещё нет:

```python
%pip install natasha==1.6.0
```

Ниже выполняется только локальное распознавание. Ответ YandexGPT берём
из сохранённого файла, поэтому новый запрос к API не требуется.

In [1]:
from pathlib import Path

from natasha import Doc, NewsEmbedding, NewsNERTagger, Segmenter

# Ноутбук открывается из папки notebooks. Из корня проекта он тоже работает.
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

source_path = (
    project_dir / "data" / "raw" / "russian-legal-ner" / "texts" / "test"
    / "test_002184.txt"
)
source = " ".join(source_path.read_text(encoding="utf-8").split())

# Берём две фразы с разными формами имени. Нормализуем только пробелы.
samples = [
    "Попова А.И. обратилась с иском к ответчику ООО «____» о защите прав потребителя.",
    "Расторгнуть договор №82-01 от 02.10.2012, заключенный между Поповой Александрой Ильиничной и ООО «____».",
]

segmenter = Segmenter()
tagger = NewsNERTagger(NewsEmbedding())

for text in samples:
    if text not in source:
        raise ValueError(f"Фраза не найдена в исходном акте: {text!r}")

    doc = Doc(text)
    doc.segment(segmenter)  # Токены входного текста.
    doc.tag_ner(tagger)      # Сущности, которые выделила модель.

    print(f"Фраза: {text}")
    print("Токены:", " | ".join(token.text for token in doc.tokens))
    print("Сущности (фрагмент, метка, начало, конец):")
    for span in doc.spans:
        # Координаты относятся к текущей фразе. Правая граница не включается.
        assert text[span.start:span.stop] == span.text
        print((span.text, span.type, span.start, span.stop))
    print()


Фраза: Попова А.И. обратилась с иском к ответчику ООО «____» о защите прав потребителя.
Токены: Попова | А | . | И | . | обратилась | с | иском | к | ответчику | ООО | « | _ | _ | _ | _ | » | о | защите | прав | потребителя | .
Сущности (фрагмент, метка, начало, конец):
('Попова А.И.', 'PER', 0, 11)
('ООО «____', 'ORG', 43, 52)

Фраза: Расторгнуть договор №82-01 от 02.10.2012, заключенный между Поповой Александрой Ильиничной и ООО «____».
Токены: Расторгнуть | договор | № | 82-01 | от | 02.10.2012 | , | заключенный | между | Поповой | Александрой | Ильиничной | и | ООО | « | _ | _ | _ | _ | » | .
Сущности (фрагмент, метка, начало, конец):
('Поповой', 'PER', 60, 67)
('Александрой Ильиничной', 'PER', 68, 90)
('ООО «____', 'ORG', 93, 102)



In [2]:
import json

# Сохранённый результат получен при обработке полного судебного акта.
response_path = (
    project_dir / "outputs" / "development"
    / "test_002184.user-prompt-rerun.json"
)
response = json.loads(response_path.read_text(encoding="utf-8"))
party = next(party for party in response["parties"] if party["role"] == "истец")

print(json.dumps(party, ensure_ascii=False, indent=2))


{
  "name": "Попова А.И.",
  "role": "истец",
  "inn": null,
  "kpp": null,
  "evidence": "Попова А.И. обратилась с иском к ответчику ООО «____» о защите прав потребителя"
}


### Что видно в выводе

- Natasha разделила `Попова А.И.` на пять токенов и объединила их в один
  фрагмент `PER`. Роль «истец» этот компонент не определяет.
- Во второй фразе Natasha разделила полное имя на два фрагмента:
  `Поповой` и `Александрой Ильиничной`. Выделение фрагмента из оригинала
  сохраняет его текст, но модель может ошибиться в его границах.
- YandexGPT вернул объект с именем, ролью и цитатой. Роль «истец»
  получена из контекста. Такой JSON нужно отдельно проверять по схеме
  и по источнику.

Это демонстрация формата и поведения двух подходов. Natasha получила две
отдельные фразы, а сохранённый ответ YandexGPT получен на полном акте.
Сравнение точности или времени работы в одинаковых условиях не проводилось.